# Kaggle Notebook: GG-SAT Model Adversarial Attack Evaluation
This notebook is fully self-contained and runs on Kaggle (with GPU accelerated environment).
It loads the custom GG-SAT pre-trained model and evaluates its robustness against:
1. Clean images
2. Dense FGSM Attack
3. Dense PGD Attack (10 iterations)
4. Sparse Top-K PGD Attack with Dynamic Masking (for various k-ratios: 0.0 to 1.0)

Multi-GPU DataParallel acceleration is automatically utilized if multiple GPUs are available.

In [6]:
import os
import time
import random
import shutil
import tarfile
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.models as tv_models
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from tqdm import tqdm

# Set seed for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [7]:
# 1. Define CIFAR-adapted ResNet-18 (Must match architecture used during GG-SAT training)
def make_cifar_resnet18(num_classes=10):
    model = tv_models.resnet18(num_classes=num_classes)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model

In [8]:
# Path to the trained model weights
model_candidates = [
    "/kaggle/input/models/truongnhatquangk18dn/default/tensorflow2/default/1/SparseRobustResNet18.pt",
    "/kaggle/input/models/truongnhatquangk18dn/ggsat/tensorflow2/default/1/SparseRobustResNet18.pt",
    "d:/Kì 8/IMP302/New folder/SparseRobustResNet18.pt",
    "d:/Kì 8/IMP302/New folder/AA-main/SparseRobustResNet18.pt",
    "../../SparseRobustResNet18.pt",
    "../SparseRobustResNet18.pt",
    "./SparseRobustResNet18.pt",
    "/kaggle/input/sparse-robust-resnet18/SparseRobustResNet18.pt"
]

model_path = None
for candidate in model_candidates:
    if os.path.exists(candidate):
        model_path = candidate
        break

if model_path is None:
    raise FileNotFoundError(
        "Could not locate 'SparseRobustResNet18.pt'. Please ensure the file exists "
        "at one of the candidate paths or update model_candidates in the code."
    )

print(f"Loading trained GG-SAT model weights from: {model_path}")
model = make_cifar_resnet18().to(device)

# Load state dict
state_dict = torch.load(model_path, map_location=device)
model.load_state_dict(state_dict)

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs with DataParallel!")
    model = nn.DataParallel(model)

model.eval()
print("Model loaded and set to evaluation mode successfully!")

Loading trained GG-SAT model weights from: /kaggle/input/models/truongnhatquangk18dn/default/tensorflow2/default/1/SparseRobustResNet18.pt
Using 2 GPUs with DataParallel!
Model loaded and set to evaluation mode successfully!


In [9]:
# 2. Define Attack Algorithms
def fgsm_attack(model, images, labels, eps=8/255):
    images = images.clone().detach().to(images.device)
    labels = labels.to(images.device)
    loss_fn = nn.CrossEntropyLoss()
    
    images.requires_grad = True
    outputs = model(images)
    loss = loss_fn(outputs, labels)
    grad = torch.autograd.grad(loss, images, retain_graph=False, create_graph=False)[0]
    
    adv_images = images + eps * grad.sign()
    adv_images = torch.clamp(adv_images, min=0, max=1).detach()
    return adv_images

def pgd_attack(model, images, labels, eps=8/255, alpha=2/255, iters=10):
    images = images.clone().detach().to(images.device)
    labels = labels.to(images.device)
    loss_fn = nn.CrossEntropyLoss()
    
    adv_images = images + torch.empty_like(images).uniform_(-eps, eps)
    adv_images = torch.clamp(adv_images, min=0, max=1).detach()
    
    for _ in range(iters):
        adv_images.requires_grad = True
        outputs = model(adv_images)
        loss = loss_fn(outputs, labels)
        grad = torch.autograd.grad(loss, adv_images, retain_graph=False, create_graph=False)[0]
        
        adv_images = adv_images.detach() + alpha * grad.sign()
        delta = torch.clamp(adv_images - images, min=-eps, max=eps)
        adv_images = torch.clamp(images + delta, min=0, max=1).detach()
    return adv_images

def topk_pgd_attack(model, images, labels, eps=8/255, alpha=2/255, iters=10, k_ratio=0.1, dynamic=True):
    images = images.clone().detach().to(images.device)
    labels = labels.to(images.device)
    loss_fn = nn.CrossEntropyLoss()
    
    adv_images = images.clone().detach()
    mask = None
    
    for t in range(iters):
        adv_images.requires_grad = True
        outputs = model(adv_images)
        loss = loss_fn(outputs, labels)
        grad = torch.autograd.grad(loss, adv_images, retain_graph=False, create_graph=False)[0]
        
        with torch.no_grad():
            if dynamic or (t == 0):
                if k_ratio == 0:
                    mask = torch.zeros_like(grad)
                else:
                    score = grad.abs()
                    score_flatten = score.view(score.size(0), -1)
                    N = score_flatten.size(1)
                    
                    k_max = int(N * k_ratio)
                    if dynamic:
                        k_t = int(k_max * (1 - t / iters))
                    else:
                        k_t = k_max
                    if k_t < 1: k_t = 1
                    
                    topk_vals, _ = torch.topk(score_flatten, k_t, dim=1)
                    tau = topk_vals[:, -1].view(-1, 1, 1, 1)
                    mask = (score >= tau).float()
                
            update = alpha * grad.sign() * mask
            adv_images.data.copy_(images + torch.clamp(adv_images + update - images, min=-eps, max=eps))
            adv_images.data.clamp_(0, 1)
            
    return adv_images

In [10]:
# 3. Load CIFAR-10 Data (Test Set Only)
def prepare_cifar10():
    target_dir = './data'
    extracted_target = os.path.join(target_dir, 'cifar-10-batches-py')
    tar_target = os.path.join(target_dir, 'cifar-10-python.tar.gz')
    
    if os.path.exists(extracted_target):
        print("CIFAR-10 is already prepared.")
        return
    
    os.makedirs(target_dir, exist_ok=True)
    candidates = [
        '/kaggle/input/datasets/truongnhatquangk18dn/aadata/cifar-10-python',
        '/kaggle/input/aadata/cifar-10-python',
        '/kaggle/input/aadata',
        '/kaggle/input/cifar-10-python'
    ]
    
    found = False
    for candidate in candidates:
        if not os.path.exists(candidate):
            continue
        
        # Check if contains extracted files directly
        if os.path.exists(os.path.join(candidate, 'batches.meta')):
            print(f"Linking extracted CIFAR-10 from {candidate}")
            os.symlink(candidate, extracted_target)
            found = True
            break
        elif os.path.exists(os.path.join(candidate, 'cifar-10-batches-py')):
            print(f"Linking extracted CIFAR-10 from {candidate}")
            os.symlink(os.path.join(candidate, 'cifar-10-batches-py'), extracted_target)
            found = True
            break
            
        # Check if tar.gz exists
        tar_path = os.path.join(candidate, 'cifar-10-python.tar.gz')
        if os.path.isfile(tar_path):
            print(f"Copying and extracting {tar_path}")
            shutil.copy(tar_path, tar_target)
            with tarfile.open(tar_target, 'r:gz') as tar:
                tar.extractall(path=target_dir)
            found = True
            break
        elif os.path.isfile(candidate) and candidate.endswith('.tar.gz'):
            print(f"Copying and extracting {candidate}")
            shutil.copy(candidate, tar_target)
            with tarfile.open(tar_target, 'r:gz') as tar:
                tar.extractall(path=target_dir)
            found = True
            break

    if not found:
        print("Warning: Could not find local CIFAR-10 dataset candidate. Will download from torchvision if internet is enabled.")

# Prepare the dataset
prepare_cifar10()

val_transform = transforms.Compose([
    transforms.ToTensor(),
])

print("Loading CIFAR-10 test dataset...")
test_set = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=val_transform)
test_loader = DataLoader(test_set, batch_size=128, shuffle=False, num_workers=2)

CIFAR-10 is already prepared.
Loading CIFAR-10 test dataset...


In [11]:
# 4. Gather 1000 test samples for final evaluation
eval_size = 1000
eval_images = []
eval_labels = []
samples_collected = 0
for img, lbl in test_loader:
    if samples_collected >= eval_size:
        break
    size_to_add = min(eval_size - samples_collected, img.size(0))
    eval_images.append(img[:size_to_add])
    eval_labels.append(lbl[:size_to_add])
    samples_collected += size_to_add

eval_images = torch.cat(eval_images, dim=0).to(device)
eval_labels = torch.cat(eval_labels, dim=0).to(device)

def evaluate_model_attack(model, loader_or_tensors, eval_size=None):
    clean_correct = 0
    fgsm_correct = 0
    pgd_correct = 0
    k_values = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
    sparse_correct = {k: 0 for k in k_values}
    total = 0
    
    if eval_size is not None:
        images, labels = loader_or_tensors
        batch_size = 128
        batches = []
        for i in range(0, labels.size(0), batch_size):
            batches.append((images[i:i+batch_size], labels[i:i+batch_size]))
    else:
        batches = loader_or_tensors
        
    for img_batch, lbl_batch in (tqdm(batches, desc="Evaluating batches") if eval_size is None else batches):
        img_batch, lbl_batch = img_batch.to(device), lbl_batch.to(device)
        batch_size = lbl_batch.size(0)
        total += batch_size
        
        # A. Clean
        with torch.no_grad():
            out_clean = model(img_batch)
            _, pred_clean = torch.max(out_clean, 1)
            clean_correct += (pred_clean == lbl_batch).sum().item()
            
        # B. FGSM
        adv_fgsm = fgsm_attack(model, img_batch, lbl_batch, eps=8/255)
        with torch.no_grad():
            out_fgsm = model(adv_fgsm)
            _, pred_fgsm = torch.max(out_fgsm, 1)
            fgsm_correct += (pred_fgsm == lbl_batch).sum().item()
            
        # C. PGD-10
        adv_pgd = pgd_attack(model, img_batch, lbl_batch, eps=8/255, alpha=2/255, iters=10)
        with torch.no_grad():
            out_pgd = model(adv_pgd)
            _, pred_pgd = torch.max(out_pgd, 1)
            pgd_correct += (pred_pgd == lbl_batch).sum().item()
            
        # D. Sparse PGD
        for k in k_values:
            adv_sparse = topk_pgd_attack(model, img_batch, lbl_batch, eps=8/255, alpha=2/255, iters=10, k_ratio=k, dynamic=True)
            with torch.no_grad():
                out_sparse = model(adv_sparse)
                _, pred_sparse = torch.max(out_sparse, 1)
                sparse_correct[k] += (pred_sparse == lbl_batch).sum().item()
                
    # Calculate Accuracies
    clean_acc = 100 * clean_correct / total
    fgsm_acc = 100 * fgsm_correct / total
    pgd_acc = 100 * pgd_correct / total
    sparse_acc = {k: 100 * sparse_correct[k] / total for k in k_values}
    
    return clean_acc, fgsm_acc, pgd_acc, sparse_acc, total

print(f"\nRunning final adversarial evaluations on {eval_size} test samples...")
clean_acc, fgsm_acc, pgd_acc, sparse_acc, total_samples = evaluate_model_attack(model, (eval_images, eval_labels), eval_size)


Running final adversarial evaluations on 1000 test samples...


In [12]:
# 5. Print Summary Report
results_summary = [
    {"Attack": "Clean (No Attack)", "Robust Accuracy": f"{clean_acc:.2f}%", "ASR": "0.00%"},
    {"Attack": "FGSM (eps=8/255)", "Robust Accuracy": f"{fgsm_acc:.2f}%", "ASR": f"{clean_acc - fgsm_acc:.2f}%"},
    {"Attack": "PGD-10 (eps=8/255)", "Robust Accuracy": f"{pgd_acc:.2f}%", "ASR": f"{clean_acc - pgd_acc:.2f}%"},
]
k_values = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
for k in k_values:
    results_summary.append({
        "Attack": f"Sparse PGD (k={k})",
        "Robust Accuracy": f"{sparse_acc[k]:.2f}%",
        "ASR": f"{clean_acc - sparse_acc[k]:.2f}%"
    })
df = pd.DataFrame(results_summary)

print("\n" + "="*60)
print(f" GG-SAT ROBUST MODEL EVALUATION REPORT ({total_samples} samples)")
print("="*60)
print(df.to_string(index=False))
print("="*60)


 GG-SAT ROBUST MODEL EVALUATION REPORT (1000 samples)
            Attack Robust Accuracy    ASR
 Clean (No Attack)          88.30%  0.00%
  FGSM (eps=8/255)          48.70% 39.60%
PGD-10 (eps=8/255)          36.20% 52.10%
Sparse PGD (k=0.0)          88.30%  0.00%
Sparse PGD (k=0.1)          73.80% 14.50%
Sparse PGD (k=0.2)          63.90% 24.40%
Sparse PGD (k=0.3)          57.60% 30.70%
Sparse PGD (k=0.4)          51.90% 36.40%
Sparse PGD (k=0.5)          48.40% 39.90%
Sparse PGD (k=0.6)          45.10% 43.20%
Sparse PGD (k=0.7)          42.70% 45.60%
Sparse PGD (k=0.8)          41.40% 46.90%
Sparse PGD (k=0.9)          40.50% 47.80%
Sparse PGD (k=1.0)          39.20% 49.10%
